# KNN-CUDA Example using Pandas dataframes

In [6]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, mean_squared_error

from knn import KNNClassifier, KNNRegressor

In [7]:
iris = load_iris()
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['target'] = iris.target
df['species'] = iris.target_names[iris.target]
print(df.head())


   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

   target species  
0       0  setosa  
1       0  setosa  
2       0  setosa  
3       0  setosa  
4       0  setosa  


In [8]:
# Train / test split — 80 / 20, stratified so each class is represented equally
X = df[iris.feature_names].values.astype(np.float32)   # [150, 4]
y = df["target"].values.astype(np.int32)               # 0=setosa, 1=versicolor, 2=virginica

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training samples : {len(X_train)}")
print(f"Test samples     : {len(X_test)}")
print(f"Features         : {X_train.shape[1]}  ({', '.join(iris.feature_names)})")
print(f"Classes          : {list(iris.target_names)}")

Training samples : 120
Test samples     : 30
Features         : 4  (sepal length (cm), sepal width (cm), petal length (cm), petal width (cm))
Classes          : [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]


## Classification — predict iris species

`KNNClassifier` sends the distance computation to the GPU (parallel) and does the top-k selection and majority vote on the CPU (sequential).

In [9]:
clf = KNNClassifier(k=5)
clf.fit(X_train, y_train)

preds = clf.predict(X_test)

# Map numeric predictions back to species names for readability
pred_names   = iris.target_names[preds]
actual_names = iris.target_names[y_test]

results_df = pd.DataFrame({
    "actual":    actual_names,
    "predicted": pred_names,
    "correct":   preds == y_test,
})
print(results_df.to_string(index=False))

    actual  predicted  correct
    setosa     setosa     True
 virginica  virginica     True
versicolor versicolor     True
versicolor versicolor     True
    setosa     setosa     True
versicolor versicolor     True
    setosa     setosa     True
    setosa     setosa     True
 virginica  virginica     True
versicolor versicolor     True
 virginica  virginica     True
 virginica  virginica     True
 virginica  virginica     True
versicolor versicolor     True
    setosa     setosa     True
    setosa     setosa     True
    setosa     setosa     True
versicolor versicolor     True
versicolor versicolor     True
 virginica  virginica     True
    setosa     setosa     True
 virginica  virginica     True
versicolor versicolor     True
 virginica  virginica     True
 virginica  virginica     True
versicolor versicolor     True
versicolor versicolor     True
    setosa     setosa     True
 virginica  virginica     True
    setosa     setosa     True


In [10]:
acc = accuracy_score(y_test, preds)
print(f"Accuracy: {acc * 100:.1f}%  ({int(acc * len(y_test))}/{len(y_test)} correct)\n")

print("Classification report:")
print(classification_report(y_test, preds, target_names=iris.target_names))

print("Confusion matrix (rows=actual, cols=predicted):")
cm = confusion_matrix(y_test, preds)
cm_df = pd.DataFrame(cm, index=iris.target_names, columns=iris.target_names)
print(cm_df)

Accuracy: 100.0%  (30/30 correct)

Classification report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00        10
   virginica       1.00      1.00      1.00        10

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30

Confusion matrix (rows=actual, cols=predicted):
            setosa  versicolor  virginica
setosa          10           0          0
versicolor       0          10          0
virginica        0           0         10


## Inspecting neighbours

`kneighbors()` returns the raw indices and distances — useful for understanding what the model actually found.

In [11]:
# Inspect the 5 nearest training neighbours for the first 5 test points
indices, distances = clf.kneighbors(X_test[:5])

for i in range(5):
    neighbour_species = iris.target_names[y_train[indices[i]]]
    print(f"Test point {i}  (actual: {iris.target_names[y_test[i]]:12s}  predicted: {iris.target_names[preds[i]]})")
    for j in range(clf.k):
        print(f"  neighbour {j+1}: train[{indices[i,j]:3d}]  {neighbour_species[j]:12s}  dist={distances[i,j]:.4f}")
    print()

Test point 0  (actual: setosa        predicted: setosa)
  neighbour 1: train[  0]  setosa        dist=0.1414
  neighbour 2: train[ 93]  setosa        dist=0.2449
  neighbour 3: train[119]  setosa        dist=0.3000
  neighbour 4: train[ 58]  setosa        dist=0.3000
  neighbour 5: train[116]  setosa        dist=0.3606

Test point 1  (actual: virginica     predicted: virginica)
  neighbour 1: train[ 98]  virginica     dist=0.2449
  neighbour 2: train[ 55]  virginica     dist=0.2828
  neighbour 3: train[ 39]  versicolor    dist=0.3000
  neighbour 4: train[ 78]  virginica     dist=0.3606
  neighbour 5: train[ 64]  versicolor    dist=0.4243

Test point 2  (actual: versicolor    predicted: versicolor)
  neighbour 1: train[ 87]  versicolor    dist=0.3873
  neighbour 2: train[ 12]  versicolor    dist=0.4583
  neighbour 3: train[ 24]  versicolor    dist=0.7211
  neighbour 4: train[ 76]  versicolor    dist=0.7874
  neighbour 5: train[ 37]  versicolor    dist=0.8367

Test point 3  (actual: vers

## Sweeping k

Accuracy can be sensitive to the choice of k. Here we try k = 1 through 15 to find the best value on this split.

In [12]:
k_values = range(1, 16)
k_results = []

for k in k_values:
    c = KNNClassifier(k=k)
    c.fit(X_train, y_train)
    acc = accuracy_score(y_test, c.predict(X_test))
    k_results.append({"k": k, "accuracy": f"{acc*100:.1f}%"})

k_df = pd.DataFrame(k_results).set_index("k")
print(k_df.T)

k            1      2       3       4       5      6      7      8       9   \
accuracy  96.7%  93.3%  100.0%  100.0%  100.0%  96.7%  96.7%  96.7%  100.0%   

k             10     11     12     13     14     15  
accuracy  100.0%  96.7%  96.7%  96.7%  96.7%  96.7%  


## Regression — predict petal length

We drop petal length from the features and ask the model to predict it from the remaining three measurements. This is a continuous target so we use `KNNRegressor`.

In [ ]:
feature_cols = ["sepal length (cm)", "sepal width (cm)", "petal width (cm)"]
target_col   = "petal length (cm)"

Xr = df[feature_cols].values.astype(np.float32)
yr = df[target_col].values.astype(np.float32)

Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.2, random_state=42)

reg = KNNRegressor(k=5)
reg.fit(Xr_train, yr_train)
yr_pred = reg.predict(Xr_test)

mse  = mean_squared_error(yr_test, yr_pred)
rmse = np.sqrt(mse)
mae  = float(np.mean(np.abs(yr_pred - yr_test)))

print(f"Predicting '{target_col}' from: {feature_cols}\n")
print(f"  RMSE : {rmse:.4f} cm")
print(f"  MAE  : {mae:.4f} cm")
print(f"  MSE  : {mse:.4f}")

Predicting 'petal length (cm)' from: ['sepal length (cm)', 'sepal width (cm)', 'petal width (cm)']

  RMSE : 0.3211 cm
  MAE  : 0.2587 cm
  MSE  : 0.1031


In [14]:
# Side-by-side: actual vs predicted for the first 15 test points
reg_df = pd.DataFrame({
    "actual (cm)":    np.round(yr_test[:15], 2),
    "predicted (cm)": np.round(yr_pred[:15], 2),
    "error (cm)":     np.round(yr_pred[:15] - yr_test[:15], 2),
})
print(reg_df.to_string(index=False))

 actual (cm)  predicted (cm)  error (cm)
         4.7            4.46       -0.24
         1.7            1.42       -0.28
         6.9            6.38       -0.52
         4.5            4.52        0.02
         4.8            4.57       -0.23
         1.5            1.55        0.05
         3.6            4.15        0.55
         5.1            5.62        0.52
         4.5            4.92        0.42
         3.9            4.10        0.20
         5.1            5.43        0.33
         1.4            1.46        0.06
         1.3            1.55        0.25
         1.5            1.43       -0.07
         1.5            1.63        0.13
